# Posture statistical pipeline (paired_test)

This notebook:
- loads data from `paired_test`
- runs `paired_hypotheses_pipeline.py`
- uses a temporary output folder (no persistent saved results)
- displays key result tables immediately

In [8]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

project_root = Path.cwd()
paired_dir = project_root / "paired_test"
input_csv = paired_dir / "Final_data_frame_filtered_enrollment_pdg_merged_fes_merged_P-N_duration.csv"
pipeline_script = paired_dir / "paired_hypotheses_pipeline.py"

temp_out = tempfile.TemporaryDirectory(prefix="paired_delta_outputs_")
out_dir = Path(temp_out.name)

print(f"Project root: {project_root}")
print(f"Input CSV exists: {input_csv.exists()} -> {input_csv}")
print(f"Pipeline exists: {pipeline_script.exists()} -> {pipeline_script}")
print(f"Temporary output dir: {out_dir}")

Project root: d:\mprgs\!VSC\articles--2026Posture
Input CSV exists: True -> d:\mprgs\!VSC\articles--2026Posture\paired_test\Final_data_frame_filtered_enrollment_pdg_merged_fes_merged_P-N_duration.csv
Pipeline exists: True -> d:\mprgs\!VSC\articles--2026Posture\paired_test\paired_hypotheses_pipeline.py
Temporary output dir: C:\Users\Radim\AppData\Local\Temp\paired_delta_outputs_l_t8dqe3


In [9]:
cmd = [
    sys.executable,
    str(pipeline_script),
    "--csv", str(input_csv),
    "--out", str(out_dir),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print("Exit code:", result.returncode)
print("\nSTDOUT:\n", result.stdout)
if result.stderr.strip():
    print("\nSTDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Pipeline failed. See the output above.")

Exit code: 0

STDOUT:
 DONE.
Input CSV: d:\mprgs\!VSC\articles--2026Posture\paired_test\Final_data_frame_filtered_enrollment_pdg_merged_fes_merged_P-N_duration.csv
Outputs dir: C:\Users\Radim\AppData\Local\Temp\paired_delta_outputs_l_t8dqe3
Derived dataset: C:\Users\Radim\AppData\Local\Temp\paired_delta_outputs_l_t8dqe3\derived_dataset_with_deltas.csv
Primary results: C:\Users\Radim\AppData\Local\Temp\paired_delta_outputs_l_t8dqe3\results_tables\PRIMARY_delta_tests_with_FDR.csv


STDERR:
 d:\mprgs\!VSC\articles--2026Posture\paired_test\paired_hypotheses_pipeline.py:276: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=groups, showfliers=True)
d:\mprgs\!VSC\articles--2026Posture\paired_test\paired_hypotheses_pipeline.py:276: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; 

In [6]:
run_report_path = out_dir / "run_report.json"
results_dir = out_dir / "results_tables"
figures_dir = out_dir / "figures"

if not run_report_path.exists():
    raise FileNotFoundError("run_report.json was not created. Execute the pipeline cell first.")

run_report = json.loads(run_report_path.read_text(encoding="utf-8"))
print("RUN REPORT:")
print(json.dumps(run_report, indent=2, ensure_ascii=False))

print("\nNumber of result CSV files:", len(list(results_dir.glob("*.csv"))))
print("Number of figures:", len(list(figures_dir.glob("*.png"))))

sorted([p.name for p in results_dir.glob("*.csv")])

RUN REPORT:
{
  "outputs_dir": "d:\\mprgs\\!VSC\\articles--2026Posture\\paired_test\\paired_delta_outputs_notebook",
  "input_csv": "d:\\mprgs\\!VSC\\articles--2026Posture\\paired_test\\Final_data_frame_filtered_enrollment_pdg_merged_fes_merged_P-N_duration.csv",
  "derived_dataset_path": "d:\\mprgs\\!VSC\\articles--2026Posture\\paired_test\\paired_delta_outputs_notebook\\derived_dataset_with_deltas.csv",
  "n_subjects": 100,
  "group_counts": {
    "TD": 51,
    "PIGD": 32,
    "PD_UNCLASSIFIED": 17
  },
  "primary_endpoints": [
    "delta_tcc_cog",
    "delta_ucc_walk_stand",
    "delta_pisa_walk_rest"
  ],
  "delta_features_for_H8": [
    "delta_tcc_cog",
    "delta_ucc_walk_stand",
    "delta_pisa_walk_rest",
    "delta_ucc_walk_rest",
    "delta_tcc_walk_rest",
    "delta_back_walk_rest"
  ]
}

Number of result CSV files: 18
Number of figures: 10


['H1_models.csv',
 'H1_terms.csv',
 'H2_gait_links.csv',
 'H3_primary_moca.csv',
 'H3_secondary_cognition.csv',
 'H4_group_coefficients_on_deltas.csv',
 'H4_group_omnibus_on_deltas.csv',
 'H5_models.csv',
 'H5_terms.csv',
 'H6_LEDD_associations.csv',
 'H7_mixedlm_models.csv',
 'H7_mixedlm_terms.csv',
 'H8_cluster_by_group.csv',
 'H8_cluster_meta.csv',
 'H8_cluster_sizes.csv',
 'H8_cluster_summary.csv',
 'PRIMARY_delta_tests_with_FDR.csv',
 'QC_posture_summary_stats.csv']

In [ ]:
def show_csv(name: str, n: int = 20):
    path = results_dir / name
    if not path.exists():
        print(f"{name}: file does not exist")
        return None
    df = pd.read_csv(path)
    print(f"\n===== {name} | shape={df.shape} =====")
    display(df.head(n))
    return df

primary = show_csv("PRIMARY_delta_tests_with_FDR.csv")
h1_terms = show_csv("H1_terms.csv")
h2 = show_csv("H2_gait_links.csv")
h3_primary = show_csv("H3_primary_moca.csv")
h4_omnibus = show_csv("H4_group_omnibus_on_deltas.csv")
h6 = show_csv("H6_LEDD_associations.csv")
h8_meta = show_csv("H8_cluster_meta.csv")

temp_out.cleanup()
print("\nTemporary output folder removed.")


===== PRIMARY_delta_tests_with_FDR.csv | shape=(3, 14) =====


,endpoint,test,n,mean,sd,median,iqr,statistic,p,normality_shapiro_p,rank_biserial,p.1,q,reject_fdr
0,delta_tcc_cog,wilcoxon_signed_rank,99,-0.637202,2.817986,-0.638,3.09100,1672.0,0.005068,0.000472,-0.324444,0.005068,0.007025,True
1,delta_ucc_walk_stand,wilcoxon_signed_rank,98,1.032929,4.522622,1.404,4.68075,1639.0,0.005319,0.000283,0.324263,0.005319,0.007025,True
2,delta_pisa_walk_rest,wilcoxon_signed_rank,100,0.137780,0.523626,0.098,0.53150,1741.0,0.007025,0.023580,0.310495,0.007025,0.007025,True



===== H1_terms.csv | shape=(9, 9) =====


,term,beta,se,p,ci_low,ci_high,model,q,reject_fdr
0,pre_rest_tcc,-0.012387,0.063993,0.846513,-0.137810,0.113036,H1:delta_tcc_cog_on_pre_rest_tcc,0.846513,False
1,age_0,0.003075,0.026545,0.907778,-0.048952,0.055102,H1:delta_tcc_cog_on_pre_rest_tcc,NaN,NaN
2,sex_bin,-0.210194,0.620957,0.734987,-1.427248,1.006860,H1:delta_tcc_cog_on_pre_rest_tcc,NaN,NaN
3,pre_stand_ucc,-0.302925,0.078660,0.000118,-0.457097,-0.148754,H1:delta_ucc_walk_stand_on_pre_stand_ucc,0.000353,True
4,age_0,0.047434,0.039073,0.224760,-0.029148,0.124015,H1:delta_ucc_walk_stand_on_pre_stand_ucc,NaN,NaN
5,sex_bin,0.566667,1.429872,0.691879,-2.235832,3.369165,H1:delta_ucc_walk_stand_on_pre_stand_ucc,NaN,NaN
6,pre_rest_pisa,-0.066261,0.071399,0.353389,-0.206201,0.073679,H1:delta_pisa_walk_rest_on_pre_rest_pisa,0.530083,False
7,age_0,0.004111,0.005281,0.436239,-0.006239,0.014462,H1:delta_pisa_walk_rest_on_pre_rest_pisa,NaN,NaN
8,sex_bin,0.044925,0.091763,0.624432,-0.134927,0.224778,H1:delta_pisa_walk_rest_on_pre_rest_pisa,NaN,NaN



===== H2_gait_links.csv | shape=(6, 13) =====


...

,outcome,n,formula,factor,n_terms,stat,df_num,df_denom,p,test,q,reject_fdr
0,delta_tcc_cog,99,delta_tcc_cog ~ C(group) + age_0 + sex_bin,C(group),2,1.691533,2.0,2.0,0.429228,Wald,0.643842,False
1,delta_ucc_walk_stand,98,delta_ucc_walk_stand ~ C(group) + age_0 + sex_bin,C(group),2,5.163549,2.0,2.0,0.075640,Wald,0.226919,False
2,delta_pisa_walk_rest,100,delta_pisa_walk_rest ~ C(group) + age_0 + sex_bin,C(group),2,0.538595,2.0,2.0,0.763916,Wald,0.763916,False



===== H6_LEDD_associations.csv | shape=(2, 11) =====


,outcome,n,formula,term,beta,se,p,ci_low,ci_high,q,reject_fdr
0,delta_ucc_walk_stand,89,delta_ucc_walk_stand ~ levodopa_equivalent + d...,levodopa_equivalent,-0.001470,0.000916,0.108387,-0.003265,0.000325,0.157984,False
1,delta_pisa_walk_rest,91,delta_pisa_walk_rest ~ levodopa_equivalent + d...,levodopa_equivalent,0.000149,0.000105,0.157984,-0.000058,0.000356,0.157984,False



===== H8_cluster_meta.csv | shape=(1, 5) =====


,status,n_clustered,features,best_k,silhouette
0,ok,97,"['delta_tcc_cog', 'delta_ucc_walk_stand', 'del...",3,0.165915
